# Week 7 · In-class boosting workbook
**AI & Machine Learning for Business · MSc Business Analytics and AI**

**Open from:** Blackboard → Week 7 → *In-class boosting workbook* → Open in Colab → **File › Save a copy in Drive**. Blackboard is the source of truth if you lose or overwrite your copy.

**Class format:** watch the instructor demonstrate the warm-up and first residual correction; then work in pairs on the marked Task cells. We will pause to compare evidence after Tasks 2 and 4. Do not simply run every cell from top to bottom.

### How every task works
| Step | What you do |
|---|---|
| **1 · Predict** | Type your prediction in the `PREDICT` cell *before* running anything. |
| **2 · Change** | Edit only the cell marked **✏️ CHANGE**. One change per run. |
| **3 · Run** | Run the cell below it. |
| **4 · Record** | Call `record(...)` with the result. Your log builds up at the bottom. |
| **5 · Decide** | Keep, reject, or follow up, with one sentence of evidence. |

| Task | Time | Slide |
|---|---|---|
| Warm-up · gradient descent (instructor demo) | during lecture | Gradient descent slides |
| Task 1 · Residual update by hand | 5 min | *Task 1 · Residual update by hand* |
| Task 2 · Learning rate and rounds | 8 min | *Task 2 · Learning rate and rounds* |
| Task 3 · Parameter laboratory | 10 min | *Task 3 · Parameter laboratory* |
| Task 4 · Model choice | 8 min | *Task 4 · Model choice from validation evidence* |
| Retrieval check | 4 min | *Retrieval check* |
| Extension · threshold under capacity | after class, optional | *From probabilities to policy* |

Everything runs on a free Colab CPU in under two minutes. The data are simulated, so nothing here is reused in any take-home exercise.

In [ ]:
# Setup — run once. Colab already includes xgboost and lightgbm; uncomment if your environment does not.
# !pip -q install xgboost lightgbm

import time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss, average_precision_score, roc_auc_score, brier_score_loss, mean_squared_error
import xgboost as xgb, lightgbm as lgb

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, TEAL, GRAY = "#2458A6", "#B7791F", "#B54740", "#0A8F8F", "#9AA3B2"

# ── Your experiment log ──────────────────────────────────────────────
LOG = []
def record(run, change, prediction, result, decision, why=""):
    # Append one row to your experiment log and show the log so far.
    LOG.append(dict(run=run, change=change, prediction=prediction, result=result, decision=decision, why=why))
    display(pd.DataFrame(LOG))
print("Ready. xgboost", xgb.__version__, "| lightgbm", lgb.__version__)

## Warm-up · Gradient descent in one dimension *(instructor demo during the lecture)*

**Picture a hiker in fog.** You cannot see the valley floor. You can only feel the slope under your feet.
So you take a step downhill, feel the slope again, and repeat. The **learning rate** is your step size.

Here the "hill" is a loss curve $L(w) = (w-3)^2$. Its slope is $L'(w) = 2(w-3)$. The best $w$ is 3.
Each step: $w \leftarrow w - \eta \cdot L'(w)$.

In [ ]:
def descend(eta, w0=-2.0, steps=12):
    path = [w0]
    for _ in range(steps):
        w = path[-1]
        path.append(w - eta * 2 * (w - 3))   # step against the slope
    return np.array(path)

def plot_descent(etas):
    grid = np.linspace(-3, 9, 200)
    fig, axes = plt.subplots(1, len(etas), figsize=(4.2 * len(etas), 3.2), sharey=True)
    for ax, eta in zip(np.atleast_1d(axes), etas):
        path = descend(eta)
        ax.plot(grid, (grid - 3) ** 2, color=GRAY)
        ax.plot(path, (path - 3) ** 2, "-o", color=BLUE, ms=4)
        ax.set_title(f"η = {eta}: w ends at {path[-1]:.2f}")
        ax.set_xlabel("w (the knob being tuned)")
    np.atleast_1d(axes)[0].set_ylabel("loss")
    plt.tight_layout(); plt.show()

plot_descent([0.05, 0.4, 1.05])

**Read the three panels:** too small a step crawls, a good step arrives quickly, and too large a step overshoots and diverges.

**Where this shows up:** logistic regression tunes its *coefficients* this way. A single tree and a random forest do **not**:
they search for good splits greedily. Gradient boosting applies the same downhill idea to the *predictions themselves*:
each new tree is one step downhill. Tasks 1 and 2 make that concrete.

In [ ]:
# Logistic regression learns its coefficients by stepping downhill on log loss.
# Two features, plain gradient descent, compared with scikit-learn.
Xd = rng.normal(size=(2000, 2)); true_w = np.array([1.5, -2.0])
yd = rng.binomial(1, 1 / (1 + np.exp(-(Xd @ true_w - 0.5))))
w, b, eta = np.zeros(2), 0.0, 0.5
for step in range(300):
    p = 1 / (1 + np.exp(-(Xd @ w + b)))
    w -= eta * Xd.T @ (p - yd) / len(yd)    # gradient of log loss w.r.t. coefficients
    b -= eta * np.mean(p - yd)
sk = LogisticRegression(C=1e6).fit(Xd, yd)
print("our gradient descent:", np.round(w, 3), round(b, 3))
print("scikit-learn        :", np.round(sk.coef_[0], 3), round(sk.intercept_[0], 3))

## Task 1 · Residual update by hand *(5 min, pairs)*

A booster's current predictions for three rows, and the next tree's output for each row.
**Fill in `my_new_F` by hand** using $F_{new} = F_{current} + \eta \cdot h(x)$ with $\eta = 0.1$, then run the check.

| Row | y | Current F(x) | Residual y − F | Tree h(x) |
|---|---|---|---|---|
| A | 1 | 0.40 | +0.60 | +0.50 |
| B | 0 | 0.30 | −0.30 | −0.20 |
| C | 1 | 0.75 | +0.25 | +0.20 |

In [ ]:
# ✏️ CHANGE — your hand calculations
my_new_F = {"A": None, "B": None, "C": None}

In [ ]:
lab = pd.DataFrame({"row": list("ABC"), "y": [1, 0, 1], "F_current": [0.40, 0.30, 0.75], "tree_h": [0.50, -0.20, 0.20]})
lab["residual"] = lab["y"] - lab["F_current"]
lab["F_new (η=0.1)"] = (lab["F_current"] + 0.1 * lab["tree_h"]).round(2)
lab["your answer"] = lab["row"].map(my_new_F)
lab["correct?"] = np.isclose(lab["your answer"].astype(float), lab["F_new (η=0.1)"])
lab

**Write one sentence:** why does η change the *size* of each correction but not its *direction*?

*Your answer:* …

## From residuals to a booster *(run, 2 min)*

The slide pseudocode, line for line in Python. Squared-error loss, so the negative gradient is simply the residual $y - F$.

In [ ]:
def boost_regression(X_tr, y_tr, X_va, y_va, M=200, eta=0.1, depth=2):
    F0 = y_tr.mean()                                    # initialize F0 from the target
    F_tr, F_va = np.full(len(y_tr), F0), np.full(len(y_va), F0)
    hist = []
    for m in range(M):                                  # for m in 1..M
        neg_grad = y_tr - F_tr                          #   negative gradient (= residual for squared error)
        h = DecisionTreeRegressor(max_depth=depth, random_state=SEED).fit(X_tr, neg_grad)  # fit a shallow tree
        F_tr += eta * h.predict(X_tr)                   #   F_m = F_{m-1} + η·h_m
        F_va += eta * h.predict(X_va)
        hist.append((mean_squared_error(y_tr, F_tr), mean_squared_error(y_va, F_va)))       # record losses
    hist = np.array(hist)
    return dict(best=int(hist[:, 1].argmin()) + 1, hist=hist)  # retain the best validated round

Xr = rng.uniform(0, 10, (1500, 3))
yr = np.sin(Xr[:, 0]) + 0.5 * Xr[:, 1] * (Xr[:, 2] > 5) + rng.normal(0, 0.3, 1500)
Xr_tr, Xr_va, yr_tr, yr_va = Xr[:1000], Xr[1000:], yr[:1000], yr[1000:]

ours = boost_regression(Xr_tr, yr_tr, Xr_va, yr_va, M=300, eta=0.1, depth=2)
skl = GradientBoostingRegressor(n_estimators=ours["best"], learning_rate=0.1, max_depth=2, random_state=SEED).fit(Xr_tr, yr_tr)
print(f"Best round: {ours['best']}")
print(f"Validation MSE — our loop: {ours['hist'][ours['best']-1, 1]:.4f} | scikit-learn: {mean_squared_error(yr_va, skl.predict(Xr_va)):.4f}")

## Task 2 · Learning rate and rounds *(8 min)*

Same data, depth-3 trees, up to 600 rounds. You will run **three** learning rates, one at a time.

For each run: **predict** the best round, **change** `LEARNING_RATE`, **run**, then **record**.

In [ ]:
# PREDICT before running: for η = 0.30, 0.10 and 0.03, roughly which round will validation loss bottom out at?
PREDICT_task2 = "type your prediction here"

In [ ]:
# ✏️ CHANGE — one value per run: 0.30, then 0.10, then 0.03
LEARNING_RATE = 0.30

In [ ]:
r = boost_regression(Xr_tr, yr_tr, Xr_va, yr_va, M=600, eta=LEARNING_RATE, depth=3)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(r["hist"][:, 0], color=GRAY, label="training MSE")
ax.plot(r["hist"][:, 1], color=BLUE, label="validation MSE")
ax.axvline(r["best"] - 1, color=RED, ls="--", lw=1)
ax.set_ylim(0, 0.4); ax.set_xlabel("boosting round"); ax.legend(frameon=False)
ax.set_title(f"η = {LEARNING_RATE}: best round {r['best']}, best validation MSE {r['hist'][r['best']-1, 1]:.4f}")
plt.show()

In [ ]:
# RECORD — copy the numbers from the chart title. Repeat for each learning rate.
record(run=f"LR={LEARNING_RATE}", change="learning rate", prediction=PREDICT_task2,
       result=f"best round {r['best']}, val MSE {r['hist'][r['best']-1, 1]:.4f}",
       decision="keep / reject / follow-up",
       why="one sentence of evidence")

## A business setting: subscription churn with a time split *(run, 2 min)*

Simulated streaming-subscription customers in 24 monthly cohorts. **Prediction moment:** 30 days before renewal.
**Action:** offer a retention discount. We train on cohorts 0–15, validate on 16–19, and keep 20–23 as a test set.

### Three metrics you will use
| Metric | Question it answers | Better is |
|---|---|---|
| **Log loss** | Are the probabilities numerically trustworthy? Confident wrong answers are punished hard. | lower |
| **PR-AUC** | Does the ranking find churners among many stayers? Baseline = churn rate. | higher |
| **Brier score** | Average squared gap between predicted probability and what happened: $\frac{1}{n}\sum (p_i - y_i)^2$ | lower |

In [ ]:
# Brier by hand: three customers
p_demo, y_demo = np.array([0.9, 0.2, 0.6]), np.array([1, 0, 0])
print("by hand     :", round(((p_demo - y_demo) ** 2).mean(), 3), " = (0.01 + 0.04 + 0.36) / 3")
print("scikit-learn:", round(brier_score_loss(y_demo, p_demo), 3))

In [ ]:
def make_churn(n=20000, seed=SEED):
    g = np.random.default_rng(seed)
    cohort = g.integers(0, 24, n)
    tenure = g.gamma(2.0, 8.0, n).clip(1, 72)
    contract = g.choice(["monthly", "annual", "bundle"], n, p=[0.55, 0.30, 0.15])
    region = g.choice(["Northeast", "South", "Midwest", "West"], n)
    bill_change = g.normal(0, 8, n)
    support_tickets = g.poisson(1.0, n)
    hours_watched = g.gamma(3, 6, n)
    devices = g.integers(1, 6, n)
    z = (-2.2 + 0.04 * cohort
         + 1.6 * (contract == "monthly") * np.exp(-tenure / 6)
         - 0.6 * (contract == "bundle")
         + 2.0 * (bill_change > 10)
         + 0.5 * np.minimum(support_tickets, 4)
         - 0.9 * (hours_watched > 15)
         + 2.0 * ((support_tickets >= 3) & (hours_watched < 8))
         - 0.15 * devices)
    churn = g.binomial(1, 1 / (1 + np.exp(-z)))
    return pd.DataFrame(dict(cohort=cohort, tenure=tenure, contract=contract, region=region, bill_change=bill_change,
                             support_tickets=support_tickets, hours_watched=hours_watched, devices=devices, churned=churn))

df = make_churn()
num = ["tenure", "bill_change", "support_tickets", "hours_watched", "devices"]; cat = ["contract", "region"]
train, valid, test = df[df.cohort <= 15], df[df.cohort.between(16, 19)], df[df.cohort >= 20]
def to_X(d):
    X = d[num + cat].copy()
    for c in cat: X[c] = X[c].astype("category")
    return X
Xtr, Xva, Xte = to_X(train), to_X(valid), to_X(test)
ytr, yva, yte = train.churned.values, valid.churned.values, test.churned.values
print(f"train {len(train):,} | valid {len(valid):,} | test {len(test):,} | churn rate in validation {yva.mean():.3f}")

## Task 3 · Parameter laboratory *(10 min)*

LightGBM on the churn data. **B0 is the reference.** For B1–B3, change exactly **one** setting from B0, predict, run, record.

| Run | Change from B0 | Question |
|---|---|---|
| B1 | learning_rate 0.05 → 0.02, max_rounds 400 → 900 | Does shrinkage improve validation? |
| B2 | num_leaves 15 → 31 | Do added interactions help? |
| B3 | min_child_samples 30 → 10 | Do smaller leaves overfit? |

In [ ]:
def fit_and_score(cfg):
    t0 = time.time()
    m = lgb.LGBMClassifier(n_estimators=cfg["max_rounds"], learning_rate=cfg["learning_rate"], num_leaves=cfg["num_leaves"],
                           min_child_samples=cfg["min_child_samples"], random_state=SEED, verbose=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])   # stop on the LATER period
    p = m.predict_proba(Xva)[:, 1]
    out = dict(best_round=m.best_iteration_, val_log_loss=round(log_loss(yva, p), 4),
               PR_AUC=round(average_precision_score(yva, p), 4), Brier=round(brier_score_loss(yva, p), 4),
               runtime_s=round(time.time() - t0, 2))
    print(cfg["run"], out); return out

B0 = dict(run="B0", learning_rate=0.05, num_leaves=15, min_child_samples=30, max_rounds=400)
b0 = fit_and_score(B0)
record("B0", "reference", "—", f"log loss {b0['val_log_loss']}, PR-AUC {b0['PR_AUC']}, best round {b0['best_round']}", "reference")

In [ ]:
# PREDICT before each run (edit the text, then run the next two cells)
PREDICT_task3 = "type your prediction here"

In [ ]:
# ✏️ CHANGE — copy B0 and change ONE setting. Do B1, then B2, then B3.
RUN = dict(B0, run="B1", learning_rate=0.02, max_rounds=900)

In [ ]:
res = fit_and_score(RUN)
record(RUN["run"], ", ".join(f"{k}={v}" for k, v in RUN.items() if k != "run" and B0[k] != v), PREDICT_task3,
       f"log loss {res['val_log_loss']}, PR-AUC {res['PR_AUC']}, best round {res['best_round']}",
       decision="keep / reject / follow-up",
       why="one sentence of evidence")

## Task 4 · Model choice from validation evidence *(8 min)*

Same rows, same time split, same metrics. Only the model and its declared configuration change.

In [ ]:
pre = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
def score(name, cfg, p, t0, best="n/a"):
    return dict(model=name, config=cfg, log_loss=round(log_loss(yva, p), 3), PR_AUC=round(average_precision_score(yva, p), 3),
                Brier=round(brier_score_loss(yva, p), 3), runtime_s=round(time.time() - t0, 1), best_round=best)
rows, P = [], {}
t0 = time.time(); m = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(train, ytr)
P["Logistic"] = m.predict_proba(valid)[:, 1]; rows.append(score("Logistic", "scaled + one-hot", P["Logistic"], t0))
t0 = time.time(); m = Pipeline([("pre", pre), ("clf", RandomForestClassifier(300, min_samples_leaf=20, n_jobs=-1, random_state=SEED))]).fit(train, ytr)
P["Random Forest"] = m.predict_proba(valid)[:, 1]; rows.append(score("Random Forest", "300 trees, min leaf 20", P["Random Forest"], t0))
t0 = time.time(); m = xgb.XGBClassifier(n_estimators=800, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                                        early_stopping_rounds=50, eval_metric="logloss", enable_categorical=True, tree_method="hist", random_state=SEED)
m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
P["XGBoost"] = m.predict_proba(Xva)[:, 1]; rows.append(score("XGBoost", "η 0.05, depth 3, sample 0.8", P["XGBoost"], t0, m.best_iteration + 1))
t0 = time.time(); m = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.05, num_leaves=15, min_child_samples=30, subsample=0.8, subsample_freq=1,
                                         colsample_bytree=0.8, random_state=SEED, verbose=-1)
m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])
P["LightGBM"] = m.predict_proba(Xva)[:, 1]; rows.append(score("LightGBM", "η 0.05, 15 leaves, min leaf 30", P["LightGBM"], t0, m.best_iteration_))
comparison = pd.DataFrame(rows).set_index("model"); comparison

In [ ]:
# High-risk segment check: 3+ support tickets and under 8 viewing hours
seg = (valid.support_tickets >= 3) & (valid.hours_watched < 8)
pd.DataFrame({"observed churn": [valid.churned[seg].mean()], **{k: [v[seg.values].mean()] for k, v in P.items()}},
             index=[f"high-risk segment (n={seg.sum()})"]).round(2)

**Write your four-line recommendation** (edit this cell):

1. **Model and exact configuration:** …
2. **Two pieces of evidence:** …
3. **One risk:** …
4. **Next check before deployment:** …

## Retrieval check *(4 min, individually, no notes)*

1. Why can training loss fall while validation loss rises?
2. Which design choice links learning rate to the number of rounds?
3. What makes a comparison between logistic regression and boosting fair?
4. Which result would make you pause deployment even when log loss improves?

## Extension · Threshold under capacity *(after class, optional)*

Offer costs \$15; a saved churner is worth \$250; an offer saves 30% of true churners. Change **one** assumption, rerun, record the best feasible threshold.

In [ ]:
# ✏️ CHANGE — one assumption at a time
COST_OFFER, VALUE_SAVED, SAVE_RATE, CAPACITY = 15, 250, 0.30, 600

In [ ]:
p = P["XGBoost"]; rows = []
for t in np.arange(0.10, 0.65, 0.05):
    flag = p >= t; tp = int((flag & (yva == 1)).sum())
    rows.append(dict(threshold=round(t, 2), offers=int(flag.sum()), churners_reached=tp,
                     expected_value=int(tp * SAVE_RATE * VALUE_SAVED - flag.sum() * COST_OFFER), feasible=flag.sum() <= CAPACITY))
policy = pd.DataFrame(rows); best = policy[policy.feasible].sort_values("expected_value").iloc[-1]
print(f"Best feasible threshold {best.threshold} → {best.offers} offers, expected value ${best.expected_value:,}"); policy

In [ ]:
record("P1", f"cost={COST_OFFER}, value={VALUE_SAVED}, capacity={CAPACITY}", "your prediction",
       f"threshold {best.threshold}, {best.offers} offers, ${best.expected_value:,}", "keep / reject / follow-up")

## Before you leave
Download your log (`File › Download › .ipynb`) or run the cell below. Submit the retrieval answers on Blackboard → Week 7 → *Retrieval check*.

In [ ]:
pd.DataFrame(LOG).to_csv("week07_experiment_log.csv", index=False)
print("Saved week07_experiment_log.csv with", len(LOG), "rows")